# Quantum Neural Networks (QNNs) — Detailed Notes (Session 14)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller  

> **Purpose.** These notes turn the slide bullets into a stand-alone reference for building, training, and evaluating quantum neural networks (QNNs) on NISQ hardware or simulators. We cover circuit design, losses, gradient computation, Qiskit ML workflows, and practical tips. Mini-exercises (with brief answers) appear at the end.

---

## Session road-map
1. Recap: feature selection → fewer qubits, better trainability  
2. What is a QNN? (parameterised circuit = model)  
3. Circuit learning: data encoding, ansatz, losses  
4. Variational quantum **classifiers** and **regressors**  
5. Training: gradients (parameter-shift), optimisers, regularisation  
6. Qiskit ML: two workflows (high-level & low-level)  
7. QNNs vs classical NNs: similarities, differences, when to use  
8. Applications & pitfalls  
9. Q&A

---

## 0) Recap — why feature selection first?
- **#qubits ≈ #features encoded.** Fewer features → shallower, less noisy circuits.  
- **Kernel/QMI ranking** (Session 11) can pre-filter features → improves QNN **sample efficiency** and **gradient signal**.  
- **Takeaway:** Do classical/quantum feature selection **before** building a QNN.

---

## 1) What is a QNN?
A **parameterised quantum circuit** (PQC) with tunable angles $\boldsymbol{\theta}$ that maps classical input $x$ to a quantum state and produces outputs via measurements.

**Structure (typical):**  
$$
|\psi(x;\boldsymbol{\theta})\rangle\;=\; U_{\text{ansatz}}(\boldsymbol{\theta})\, U_\phi(x)\,|0\rangle^{\otimes n}.
$$
- $U_\phi(x)$: **feature map** (data encoding) — e.g., $R_Z(\alpha x_i)$, entangling ZZ terms  
- $U_{\text{ansatz}}(\boldsymbol{\theta})$: trainable layers — e.g., `TwoLocal` (RY/CX) repeated $p$ times  
- Outputs: expectation(s) $\langle P \rangle$ (classifier) or $\langle \sum w_i P_i \rangle$ (regressor)

**Why QNNs on NISQ?** Shallow variational form; hybrid optimisation keeps quantum depth small.

---

## 2) Quantum circuit learning — the pieces

### 2.1 Data encoding (feature map)
- **Angle encoding:** $R_Z(\alpha x_i)$, $R_Y(\alpha x_i)$ per qubit; optionally entangle with $e^{-i\beta x_i x_j Z_i Z_j}$.  
- **Data re-uploading:** repeat $U_\phi(x)$ between ansatz layers to increase expressivity without more qubits.  
- **Normalisation:** scale inputs so angles lie in a reasonable range (e.g., $[-\pi,\pi]$).

### 2.2 Ansatz design
- **Hardware-efficient** (e.g., RY/CX layers matching device connectivity) — shallow, trainable.  
- **Problem-inspired** (e.g., symmetries, conserved quantities) — better inductive bias, fewer params.  
- **Depth/width trade-off:** more layers ↑ expressivity **but** ↑ barren plateau risk/noise.

### 2.3 Readout & loss
- **Classifier:** map $\hat y=\frac{1-\langle Z_0\rangle}{2}\in[0,1]$; loss: BCE/hinge.  
- **Regressor:** $f(x)=\sum_i w_i\langle Z_i\rangle$; loss: MSE.  
- **Multiclass:** allocate $k$ readout qubits (one-vs-rest) or encode logits and softmax classically.

---

## 3) Variational quantum classifiers (VQC)

**Pipeline**
1. Encode $x$ → state via $U_\phi(x)$  
2. Apply ansatz $U(\boldsymbol{\theta})$  
3. Measure $\langle Z_0\rangle$ (or vector of observables)  
4. Compute loss vs. label $y$, update $\boldsymbol{\theta}$

**Design tips**
- Start with **RY + CX** (or RY–RZ + CX) with 1–3 reps.  
- Entangle according to hardware coupling map; avoid all-to-all on sparse devices.  
- **Bias/variance control:** L2 penalty on $\boldsymbol{\theta}$; early stopping; shot-aware validation.

---

## 4) Variational quantum regressors (VQR)
- Same structure, but target $t\in\mathbb{R}$.  
- Output a continuous expectation; squash or rescale if target lies outside $[-1,1]$.  
- Shots noise sets a **precision floor**; average more shots or use readout calibration.

---

## 5) Training QNNs

### 5.1 Gradients: parameter-shift rule
For gates $e^{-i\theta_k P/2}$ with Pauli $P$,
$$
\frac{\partial}{\partial \theta_k}\langle H\rangle
= \tfrac{1}{2}\Big( \langle H\rangle_{\theta_k+\pi/2} - \langle H\rangle_{\theta_k-\pi/2}\Big).
$$
- **No finite differences**, exact for many native rotations.  
- Each gradient component costs **two** circuit evaluations (× shots).

### 5.2 Optimisers
- **Gradient-based:** Adam / SGD (needs parameter-shift).  
- **Gradient-free:** SPSA (2 evals/step), COBYLA (few params), Nelder–Mead.  
- **Scheduling:** cosine/step LR; mini-batches for large datasets.

### 5.3 Trainability & regularisation
- **Barren plateaus:** mitigate via shallow depth, local cost functions, layer-wise training, good init (small $\theta$), feature selection.  
- **Noise:** readout calibration, dynamical decoupling, zero-noise extrapolation; modest reps.

---

## 6) Qiskit Machine Learning — two workflows

### 6.1 High-level: `VQC` / `VQR` (conceptual)
```python
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit.algorithms.optimizers import SPSA
from qiskit_machine_learning.algorithms import VQC  # or VQR
from qiskit_aer.primitives import Estimator

feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
ansatz = TwoLocal(num_qubits=2, rotation_blocks='ry', entanglement_blocks='cx', reps=2)
optimizer = SPSA(maxiter=150)

vqc = VQC(feature_map=feature_map, ansatz=ansatz,
          optimizer=optimizer, estimator=Estimator(shots=2048))
vqc.fit(X_train, y_train)      # y_train in {0,1}
acc = vqc.score(X_test, y_test)
```

### 6.2 Low-level: `EstimatorQNN` + `NeuralNetworkClassifier`
```python
from qiskit.circuit.library import TwoLocal
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_aer.primitives import Estimator
from qiskit import QuantumCircuit
import numpy as np

# Build encoding + ansatz circuit with classical parameters as inputs
n = 2
enc = QuantumCircuit(n)
enc.ry(np.pi * 0.5, 0)  # placeholder; in practice bind per-sample
enc.ry(np.pi * 0.5, 1)

ans = TwoLocal(n, ['ry','rz'], 'cx', reps=2)
qc = enc.compose(ans)

qnn = EstimatorQNN(circuit=qc, input_params=enc.parameters,
                   weight_params=ans.parameters, estimator=Estimator(shots=2048))

clf = NeuralNetworkClassifier(qnn, optimizer=SPSA(maxiter=150))
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
```

> **Note.** APIs evolve; concept is stable: *feature map → ansatz → Estimator/primitive → optimiser*.

---

## 7) QNNs vs classical NNs — quick compare

| Aspect | Classical NN | QNN |
|---|---|---|
| Parameters | Often 10⁴–10⁹ | 10–10³ (NISQ) |
| Non-linearity | ReLU/gelu, etc. | Interference/measurement + entanglement |
| Training | Backprop, automatic differentiation | Parameter-shift / SPSA (shot-based) |
| Data scale | Large | Small/medium best |
| Hardware | GPUs/TPUs | Quantum processors/simulators |
| When to prefer | Big data, deep models | Small complex patterns; hybrid feature extraction; research on quantum advantage |

---

## 8) Applications & patterns

- **Small image/medical sets:** use 4–8 qubits; data re-uploading; hybrid classical pre-CNN → QNN head.  
- **Anomaly detection:** one-class QNN (threshold on $\langle Z\rangle$).  
- **Scientific regression:** model low-dim signals with VQR; uncertainty via shot variance.  
- **Hybrid pipelines:** classical feature selection → QNN → classical post-classifier.

---

## 9) Practical pitfalls & debugging checklist

- **Encoding range off:** angles saturate → flat loss. *Fix:* standardise inputs, tune scaling constants.  
- **Too deep ansatz:** vanishing gradients. *Fix:* fewer reps; layer-wise growth.  
- **Mismatch with hardware topology:** many SWAPs added. *Fix:* transpile with initial layout; topology-aware entanglement.  
- **Shot starvation:** noisy/unstable gradients. *Fix:* increase shots gradually; use SPSA; average mini-batches.  
- **Label leakage:** normalising across train+test. *Fix:* fit scalers on train only.

---

## 10) Worked example — binary classification (moons)

```python
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit.algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC
from qiskit_aer.primitives import Estimator

# 1) data
X, y = make_moons(n_samples=200, noise=0.1, random_state=0)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

# 2) model
feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
ansatz = TwoLocal(2, 'ry', 'cx', reps=2)
vqc = VQC(feature_map, ansatz, optimizer=COBYLA(maxiter=100),
          estimator=Estimator(shots=2048))

# 3) train & eval
vqc.fit(Xtr, ytr)
print("Test accuracy:", vqc.score(Xte, yte))
```

**Tweaks:** try `SPSA` for noisy hardware; add readout mitigation; reduce reps if unstable.

---

## 11) Mini-exercises (answers in Appendix)

1. **Parameter-shift derivation:** For a single-parameter gate $e^{-i\theta X/2}$, prove the parameter-shift gradient rule for $\langle Z_0\rangle$.  
2. **Design a 4-qubit QNN** for 3-class classification with one-vs-rest readouts; propose a loss and output mapping.  
3. **Barren plateau check:** Show that random initialisation with many layers makes $\operatorname{Var}[\nabla_\theta L]$ small; suggest two mitigations.  
4. **Shot budgeting:** For BCE loss with target margin 0.1, estimate the minimum shots per iteration so the standard error in $\hat{\langle Z\rangle}$ is ≤ 0.05.  
5. **Noise-aware transpilation:** Explain why topology-aware entanglement reduces SWAP count and improves gradient SNR.

---

## 12) Summary (Session 12)
- A QNN = $U(\theta)U_\phi(x)$ + measurement → differentiable model trained with hybrid optimisation.  
- Good **encoding** and **lightweight ansatz** matter more than depth on NISQ.  
- Gradients via **parameter-shift** or **SPSA**; mitigate noise with readout calibration, DD, ZNE.  
- Feature selection (prev. session) is a powerful prerequisite for improved trainability and accuracy.  
- Qiskit ML provides high- and low-level APIs for rapid prototyping.

---

## 13) Looking ahead
- **Next Session:** Hybrid Methods — combining QNNs with classical architectures (data re-uploading, QCNNs, QGANs).  
- **Homework 4:**  
  - Implement VQC on moons; report accuracy vs reps (1–3) and shots (256–4096).  
  - Compare SPSA vs Adam on noisy simulator.  
  - Bonus: add feature selection (top-k) and quantify impact.

---

## Appendix — solutions (sketch)

1. **Param-shift:** For $U = e^{-i\theta X/2}$, use $X$ eigenvalues ±1 and linearity to show  
   $\partial_\theta \langle H\rangle = [\langle H\rangle_{\theta+\pi/2} - \langle H\rangle_{\theta-\pi/2}]/2$.  
2. **3-class:** Use 2 readout qubits; train three one-vs-rest sigmoid heads: $p_c=\sigma(a_c)$, loss $L=\sum_c \text{BCE}(y_c,p_c)$. Predict argmax $p_c$.  
3. **Barren plateaus:** Random deep circuits approximate 2-designs; gradients concentrate near 0. Mitigate with shallow reps, layer-wise growth, local cost, structured init.  
4. **Shots:** For Bernoulli with true $p$, $\text{SE}=\sqrt{p(1-p)/S}$. Worst case $p=0.5$: need $S\ge 0.25/0.05^2=100$ shots per expectation; multiply by #terms.  
5. **Transpilation:** Entangle only neighbouring qubits to avoid SWAP insertion; fewer two-qubit errors → higher fidelity and better gradient SNR.

